---
**Hierarchical Clustering & PCA in Python**
Data Analysis Course · Week 4
---

This notebook is the Python equivalent of the R Markdown `_03_unsupervised_learning.Rmd`.
Topics: **hierarchical clustering** (correlation-based distance, dendrograms, heatmaps) and
**Principal Component Analysis (PCA)** (scree plots, scores, loadings).

Work through it cell by cell — run each code cell with **Shift+Enter**.

**Required packages:** `pandas`, `numpy`, `matplotlib`, `seaborn`, `scipy`, `scikit-learn`, `pyreadr`
```
pip install pandas numpy matplotlib seaborn scipy scikit-learn pyreadr
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import pyreadr, urllib.request

## 0 – Recap of the last sheet

Last week we kept working with the diabetes dataset, learning to measure centrality and correlation,
then moved to unsupervised learning with k-means clustering on a gene expression dataset.

## 1 – Objectives of this week

This week concludes unsupervised learning with **hierarchical clustering** and **PCA**.

## 2 – Hierarchical clustering

We work with a leukemia gene expression dataset: two types of leukemia, ALL (acute lymphoid) and
AML (acute myeloid). We have one file with gene expression and one with clinical annotations.

In [ ]:
#### This is all pre-processing of the ALL/AML dataset — RUN THIS CELL AT THE BEGINNING

urllib.request.urlretrieve(
    "https://www.dropbox.com/scl/fi/kfnxs5ltldmfj6px1sp13/all.aml.anno.rds?rlkey=yhahfyk4h240fbt1vctt3bp5b&dl=1",
    "all_aml_anno.rds"
)
urllib.request.urlretrieve(
    "https://www.dropbox.com/scl/fi/grt6yud9yjmwqx3zgwhte/all.aml.exp.rds?rlkey=hs8p83tkxvy27t8v38tw51x7i&dl=1",
    "all_aml_exp.rds"
)
all_aml_anno = pyreadr.read_r("all_aml_anno.rds")[None]
all_aml_exp = pyreadr.read_r("all_aml_exp.rds")[None]

### Using the most variable, thus informative genes

In [ ]:
# R: topVar = apply(all.aml.exp, 1, var); summary(topVar)
top_var = all_aml_exp.var(axis=1)
top_var.describe()

In [ ]:
# R: q75 = quantile(topVar, probs = 0.75)
q75 = top_var.quantile(0.75)
q75

In [ ]:
# R: i.topvar = which(topVar >= q75); all.aml.exp.topVar = all.aml.exp[i.topvar,]
i_topvar = top_var[top_var >= q75].index
all_aml_exp_topvar = all_aml_exp.loc[i_topvar]
all_aml_exp_topvar.shape   # R: dim(all.aml.exp.topVar)

### Computing the correlation between all patients (columns)

In [ ]:
# R: cor.mat = cor(all.aml.exp.topVar, method="pearson")
cor_mat = all_aml_exp_topvar.corr(method="pearson")

Let's display the correlation matrix as a heatmap. `seaborn.clustermap` is the Python equivalent of R's `pheatmap` — it draws the heatmap *and* the row/column dendrograms together.

In [ ]:
# R: library(pheatmap); pheatmap(cor.mat)
sns.clustermap(cor_mat, cmap="vlag", figsize=(8, 8))
plt.show()

# Each cell represents the correlation between the sample in the row and the sample in the column.
# The correlation of a sample to itself is always 1 (diagonal).

The color scale is biased by the 1's on the diagonal. Since these values are trivial, we remove them to avoid distorting the color scale:

In [ ]:
# R: diag(cor.mat) <- NA
cor_mat_nodiag = cor_mat.copy()
np.fill_diagonal(cor_mat_nodiag.values, np.nan)

sns.clustermap(cor_mat_nodiag, cmap="vlag", figsize=(8, 8))
plt.show()

### Plotting the dendrogram along with clinical annotations

In [ ]:
all_aml_anno.head()

In [ ]:
# R: pheatmap(cor.mat, annotation_row = all.aml.anno, annotation_col = all.aml.anno)
# seaborn.clustermap uses row_colors/col_colors for annotation tracks, mapped from categories to colors
annotation_col = all_aml_anno.iloc[:, 0]
lut = dict(zip(annotation_col.unique(), sns.color_palette("Set2", annotation_col.nunique())))
colors = annotation_col.map(lut)

sns.clustermap(cor_mat_nodiag, cmap="vlag", figsize=(10, 8), row_colors=colors, col_colors=colors)
plt.show()

# How would you interpret this dendrogram? How many clusters do you observe? Do they make clinical sense?

> `sns.clustermap` accepts a `method` parameter for alternative linkage methods (e.g. `"average"`,
> `"complete"`, `"ward"`) — try a few and see how the dendrogram topology changes!
>
> Can you redo this analysis using all genes instead of the most variable ones? Do you see a difference?

## 3 – Dimensionality reduction: PCA

To illustrate PCA, we use a different dataset describing pathological features of breast cancer
tumor samples ([description here](https://www.geeksforgeeks.org/machine-learning/breast-cancer-wisconsin-diagnostic-dataset/)).

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data"
bc_data = pd.read_csv(url, header=None)

bc_data.columns = [
    "ID", "Diagnosis",
    "radius_mean", "texture_mean", "perimeter_mean", "area_mean",
    "smoothness_mean", "compactness_mean", "concavity_mean",
    "concave_points_mean", "symmetry_mean", "fractal_dimension_mean",
    "radius_se", "texture_se", "perimeter_se", "area_se",
    "smoothness_se", "compactness_se", "concavity_se",
    "concave_points_se", "symmetry_se", "fractal_dimension_se",
    "radius_worst", "texture_worst", "perimeter_worst", "area_worst",
    "smoothness_worst", "compactness_worst", "concavity_worst",
    "concave_points_worst", "symmetry_worst", "fractal_dimension_worst"
]

bc_data = bc_data.drop(columns=["ID"])

diagnosis = bc_data["Diagnosis"].map({"M": "Malignant", "B": "Benign"})
bc_data = bc_data.drop(columns=["Diagnosis"])

The `diagnosis` Series indicates whether the patient had a malignant or benign tumor.

**Important:** patients are in rows, features are in columns — same layout scikit-learn's `PCA` expects.

### Running the PCA

In [ ]:
# R: prcomp(bc_data, center = TRUE, scale = TRUE)
# center=TRUE, scale=TRUE in R = standardize each column first (mean 0, sd 1)
from sklearn.preprocessing import StandardScaler

bc_scaled = StandardScaler().fit_transform(bc_data)
pca = PCA()
pca_scores = pca.fit_transform(bc_scaled)
print(pca_scores.shape)   # R: dim(pca_result$x)

# How many principal components do you obtain? Compare this to the shape of bc_data!

Principal components are ranked by the variance they explain — visualized with a **scree plot**:

In [ ]:
# R: var_explained <- pca_result$sdev^2; pve <- var_explained / sum(var_explained)
pve = pca.explained_variance_ratio_

plt.bar(range(1, 11), pve[:10] * 100, color="steelblue")
plt.plot(range(1, 11), pve[:10] * 100, color="red", marker="o")
plt.xlabel("Principal Component")
plt.ylabel("Variance Explained (%)")
plt.title("Scree Plot")
plt.show()

# The amount of explained variance decreases with each component.

### Plotting the patients

In [ ]:
# R: plot(pca_result$x[,1], pca_result$x[,2], col=colors[as.numeric(Diagnosis)], ...)
colors = diagnosis.map({"Malignant": "red", "Benign": "blue"})

plt.scatter(pca_scores[:, 0], pca_scores[:, 1], c=colors, s=15)
plt.xlabel(f"PC1 ({pve[0]*100:.1f}%)")
plt.ylabel(f"PC2 ({pve[1]*100:.1f}%)")
plt.title("PCA Scores Plot Colored by Diagnosis")
handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=c, label=l)
           for l, c in [("Malignant", "red"), ("Benign", "blue")]]
plt.legend(handles=handles)
plt.show()

# Redo the plot with other PCs (e.g. PC1 vs. PC3).

### Understanding the principal components

What do the principal components describe? We look at the **loadings** — the contribution of each original variable to each PC, stored in `pca.components_` (transposed compared to R's `rotation` matrix).

In [ ]:
# R: loadings_pc1 <- pca_result$rotation[, 1]
loadings = pd.DataFrame(
    pca.components_[:2].T,
    index=bc_data.columns,
    columns=["PC1", "PC2"]
)
loadings

In [ ]:
# PC1 loadings bar plot
fig, axes = plt.subplots(1, 2, figsize=(12, 8))

pc1_sorted = loadings["PC1"].sort_values()
axes[0].barh(pc1_sorted.index, pc1_sorted.values,
             color=["blue" if v > 0 else "red" for v in pc1_sorted.values])
axes[0].axvline(0, color="black")
axes[0].set_title("PC1 Loadings")
axes[0].tick_params(axis="y", labelsize=7)

pc2_sorted = loadings["PC2"].sort_values()
axes[1].barh(pc2_sorted.index, pc2_sorted.values,
             color=["blue" if v > 0 else "red" for v in pc2_sorted.values])
axes[1].axvline(0, color="black")
axes[1].set_title("PC2 Loadings")
axes[1].tick_params(axis="y", labelsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# R: fviz_pca_var(pca_result, geom = c("point","text")) — variable correlation circle
fig, ax = plt.subplots(figsize=(7, 7))
for i, var in enumerate(bc_data.columns):
    ax.arrow(0, 0, pca.components_[0, i], pca.components_[1, i],
              head_width=0.01, color="grey", alpha=0.6)
    ax.text(pca.components_[0, i] * 1.1, pca.components_[1, i] * 1.1, var, fontsize=6)
circle = plt.Circle((0, 0), 1, fill=False, linestyle="--")
ax.add_artist(circle)
ax.set_xlim(-1.1, 1.1)
ax.set_ylim(-1.1, 1.1)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
plt.show()

# How do you interpret the direction the arrows point in?

---
## Exercises

### Exercise 1 — Hierarchical clustering and k-means

1. Based on the heatmap from the correlation matrix, run k-means clustering on the patients with a
   suitable number of clusters (**note**: `all_aml_exp` has samples as columns, so transpose it first!)
   `KMeans(n_clusters=k).fit(all_aml_exp.T)`
2. Add the cluster assignment as a new column `cluster` in `all_aml_anno`.
3. Redo the heatmap so that cluster membership appears as an annotation track.
4. Repeat with other values of k.

In [ ]:
# Your code here:

### Exercise 2 — Hierarchical clustering

We computed a correlation matrix in section 2 and used it to build a clustering tree.

1. Create a 4×4 distance matrix (with your own values); remember a distance matrix must be
   **symmetrical**!

```python
# example from the lecture slides
d = pd.DataFrame(
    [[0, 2, 3, 5], [2, 0, 1, 4.1], [3, 1, 0, 4], [5, 4.1, 4, 0]],
    index=list("abcd"), columns=list("abcd")
)
sns.clustermap(d)
plt.show()
```
Try different linkage `method`s (e.g. `"single"`, `"complete"`, `"average"`, `"ward"`) to see if the
dendrogram topology changes!

2. Try building a distance matrix that gives a *different* dendrogram topology depending on the
   linkage method used. Show the dendrograms for each method.

In [ ]:
# Your code here:

### Exercise 3 — PCA

1. Redo the PCA on the breast cancer dataset, but only using the `_mean` columns.
2. Same, but using only the `_worst` columns.

In [ ]:
# Your code here:

### Going further *(expert)*

Instead of running k-means on the whole gene expression matrix, we can run it on the space where
each patient is represented by their first k principal components (`pca_scores`).

1. Run k-means with different numbers of clusters (1–10) using the first 2, 4, 6, ... principal
   components. Use the elbow method to evaluate how WSS (`.inertia_`) evolves. What is the optimal
   number of clusters?
2. Plot the patients in the PCA plane as before, but colored by their k-means cluster (k=2).

In [ ]:
# Your code here:

## Summary: What have we learned?

| R | Python | Purpose |
|---|--------|---------|
| `apply(m, 1, var)` | `df.var(axis=1)` | Row-wise variance |
| `pheatmap(cor.mat)` | `sns.clustermap(cor_mat)` | Heatmap + dendrogram |
| `diag(m) <- NA` | `np.fill_diagonal(m.values, np.nan)` | Blank out the diagonal |
| `prcomp(x, center=T, scale=T)` | `StandardScaler()` + `PCA().fit_transform()` | PCA (correlation-based) |
| `pca$sdev^2 / sum(...)` | `pca.explained_variance_ratio_` | Proportion of variance explained |
| `pca$x` | `pca.fit_transform(x)` (the scores) | PCA scores (coordinates) |
| `pca$rotation` | `pca.components_.T` | PCA loadings |
| `fviz_pca_var()` | manual `ax.arrow()` correlation circle | Variable correlation circle |
| `kmeans(x, centers=k)` | `KMeans(n_clusters=k).fit(x)` | K-means clustering |